# Day 3 — Bias-Variance & Diagnosing Model Fit

# Topic 1: Underfitting vs Overfitting

A machine learning model can fail because it is either too simple or too complex.

## Underfitting

Underfitting happens when a model is too simple to capture the important patterns in the data.

Typical symptoms:

* Poor performance on the training data.
* Poor performance on the validation data.
* The model has high bias.

Possible fixes:

* Add more useful features.
* Use a more powerful model.
* Increase the model complexity.

## Overfitting

Overfitting happens when a model is too complex and learns the training data too closely.

Typical symptoms:

* Very good performance on the training data.
* Much worse performance on the validation data.
* The model has high variance.

Possible fixes:

* Simplify the model.
* Reduce model complexity.
* Add more training data.
* Use regularization.

The main goal is to build a model that performs well on both training and unseen data.


In [4]:
import pandas as pd

data = {
    "Age": [18,19,20,21,22,20,23,19,24,21,18,22,20,23,19,24,21,20,22,23],
    "Income": [2500,3000,3500,2200,4000,2800,4500,2600,5000,3200,
               2300,3800,2900,4200,2700,4800,3100,2500,3900,4400],
    "StudyHours": [2,4,5,1,6,3,7,2,8,4,1,6,2,7,3,8,5,2,6,7],
    "City": [
        "Hebron","Nablus","Ramallah","Hebron","Ramallah",
        "Nablus","Ramallah","Hebron","Ramallah","Nablus",
        "Hebron","Ramallah","Nablus","Ramallah","Hebron",
        "Ramallah","Nablus","Hebron","Ramallah","Nablus"
    ],
    "Passed": [0,1,1,0,1,0,1,0,1,1,0,1,0,1,0,1,1,0,1,1]
}

df = pd.DataFrame(data)

print(df.head())

   Age  Income  StudyHours      City  Passed
0   18    2500           2    Hebron       0
1   19    3000           4    Nablus       1
2   20    3500           5  Ramallah       1
3   21    2200           1    Hebron       0
4   22    4000           6  Ramallah       1


In [5]:
X = df.drop("Passed", axis=1)
y = df["Passed"]

X = pd.get_dummies(
    X,
    columns=["City"],
    dtype=int
)

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (16, 6)
X_test: (4, 6)


In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Create a decision tree
model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

# Predictions
train_pred = model.predict(X_train)
val_pred = model.predict(X_test)

# Scores
train_score = accuracy_score(y_train, train_pred)
val_score = accuracy_score(y_test, val_pred)

print("Training Accuracy:", train_score)
print("Validation Accuracy:", val_score)

Training Accuracy: 1.0
Validation Accuracy: 1.0


# Topic 2: The Bias-Variance Trade-off

Bias and variance describe two different sources of model error.

## Bias

Bias is the error caused by making overly simple assumptions about the data.

A high-bias model is usually too simple and may underfit the data.

## Variance

Variance is the error caused by a model being too sensitive to the specific training data.

A high-variance model is usually too complex and may overfit the data.

## The Trade-off

As model complexity increases:

* Bias usually decreases.
* Variance usually increases.

As model complexity decreases:

* Bias usually increases.
* Variance usually decreases.

The goal is to find a balance between bias and variance.

A good model should be complex enough to learn the important patterns but simple enough to generalize to unseen data.


In [8]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

depths = [1, 2, 3, 5, None]

for depth in depths:

    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_test)

    train_score = accuracy_score(y_train, train_pred)
    val_score = accuracy_score(y_test, val_pred)

    print(
        f"max_depth={depth} | "
        f"Train={train_score:.2f} | "
        f"Validation={val_score:.2f}"
    )

max_depth=1 | Train=1.00 | Validation=1.00
max_depth=2 | Train=1.00 | Validation=1.00
max_depth=3 | Train=1.00 | Validation=1.00
max_depth=5 | Train=1.00 | Validation=1.00
max_depth=None | Train=1.00 | Validation=1.00


# Topic 3: Diagnosing Model Fit with the Train-Validation Gap

The difference between training and validation performance is an important diagnostic tool.

We can use the train-validation gap to identify the type of model problem.

| Training Score | Validation Score | Diagnosis    |
| -------------- | ---------------- | ------------ |
| Low            | Low              | Underfitting |
| High           | Much lower       | Overfitting  |
| High           | High             | Good Fit     |

## Underfitting

When both training and validation scores are low, the model is probably too simple.

## Overfitting

When the training score is high but the validation score is much lower, the model is probably memorizing the training data.

## Good Fit

When both scores are high and the gap between them is small, the model is likely generalizing well.

The goal is not simply to maximize the training score. The goal is to achieve strong validation performance with a small train-validation gap.


In [9]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

model = DecisionTreeClassifier(
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

train_pred = model.predict(X_train)
val_pred = model.predict(X_test)

train_score = accuracy_score(y_train, train_pred)
val_score = accuracy_score(y_test, val_pred)

gap = train_score - val_score

print("Training Score:", train_score)
print("Validation Score:", val_score)
print("Train-Validation Gap:", gap)

if train_score < 0.70 and val_score < 0.70:
    print("Diagnosis: Underfitting")

elif train_score > 0.90 and gap > 0.10:
    print("Diagnosis: Overfitting")

else:
    print("Diagnosis: Reasonable Fit")

Training Score: 1.0
Validation Score: 1.0
Train-Validation Gap: 0.0
Diagnosis: Reasonable Fit


# Topic 4: Regularization

Regularization is a technique used to reduce overfitting.

It adds a penalty for model complexity and discourages the model from relying too heavily on individual features.

Two common types of regularization are:

## Ridge Regression — L2

Ridge uses L2 regularization.

It shrinks model coefficients toward zero but normally does not make them exactly zero.

```text
Ridge = L2 Regularization
```

## Lasso Regression — L1

Lasso uses L1 regularization.

It can shrink some feature coefficients exactly to zero.

This means Lasso can also perform a form of feature selection.

```text
Lasso = L1 Regularization
```

## The Alpha Parameter

The `alpha` parameter controls the strength of the regularization.

A larger `alpha` applies a stronger penalty and produces a simpler model.

Choosing a good value for `alpha` is a hyperparameter tuning problem.


In [10]:
#Code — Ridge

from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)

ridge.fit(X_train, y_train)

ridge_train_score = ridge.score(X_train, y_train)
ridge_test_score = ridge.score(X_test, y_test)

print("Ridge Training Score:", ridge_train_score)
print("Ridge Test Score:", ridge_test_score)

Ridge Training Score: 0.8383638402823819
Ridge Test Score: 0.8346607844102747


In [11]:
#Code — Lasso

from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.1)

lasso.fit(X_train, y_train)

lasso_train_score = lasso.score(X_train, y_train)
lasso_test_score = lasso.score(X_test, y_test)

print("Lasso Training Score:", lasso_train_score)
print("Lasso Test Score:", lasso_test_score)


Lasso Training Score: 0.6653649700814733
Lasso Test Score: 0.5140790905802761


# Day 3 — Hands-On Task

# Task Step 1: Deliberately Overfit a Model

We will intentionally create an overfitted model using a very deep Decision Tree.

A deep tree can become too complex and memorize the training data.

We expect:

* Very high training accuracy.
* Lower validation accuracy.
* A large train-validation gap.

This gap will provide evidence that the model is overfitting.


In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Intentionally complex model
overfit_model = DecisionTreeClassifier(
    max_depth=None,
    random_state=42
)

# Train the model
overfit_model.fit(X_train, y_train)

# Predictions
train_pred = overfit_model.predict(X_train)
val_pred = overfit_model.predict(X_test)

# Scores
overfit_train_score = accuracy_score(
    y_train,
    train_pred
)

overfit_val_score = accuracy_score(
    y_test,
    val_pred
)

overfit_gap = overfit_train_score - overfit_val_score

print("Training Accuracy:", overfit_train_score)
print("Validation Accuracy:", overfit_val_score)
print("Train-Validation Gap:", overfit_gap)

Training Accuracy: 1.0
Validation Accuracy: 1.0
Train-Validation Gap: 0.0


# Task Step 2: Deliberately Underfit a Model

We will intentionally create an overly simple Decision Tree.

A tree with a very small depth has limited ability to learn complex patterns.

We expect:

* Low training accuracy.
* Low validation accuracy.
* A relatively small train-validation gap.

This indicates that the model is underfitting because it is too simple to capture the patterns in the data.


In [13]:
# Intentionally simple model
underfit_model = DecisionTreeClassifier(
    max_depth=1,
    random_state=42
)

# Train the model
underfit_model.fit(X_train, y_train)

# Predictions
train_pred = underfit_model.predict(X_train)
val_pred = underfit_model.predict(X_test)

# Scores
underfit_train_score = accuracy_score(
    y_train,
    train_pred
)

underfit_val_score = accuracy_score(
    y_test,
    val_pred
)

underfit_gap = underfit_train_score - underfit_val_score

print("Training Accuracy:", underfit_train_score)
print("Validation Accuracy:", underfit_val_score)
print("Train-Validation Gap:", underfit_gap)

Training Accuracy: 1.0
Validation Accuracy: 1.0
Train-Validation Gap: 0.0


# Task Step 3: Fix the Overfitted Model

The previous model was intentionally allowed to grow without a depth limit.

We will now reduce the model complexity by setting a maximum tree depth.

A simpler model should generalize better and reduce the train-validation gap.

We will compare the original overfitted model with the simpler model.

The goal is not necessarily to maximize the training score.

The goal is to improve validation performance while reducing the gap between training and validation scores.


In [14]:
# Reduced-complexity model
fixed_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

# Train the model
fixed_model.fit(X_train, y_train)

# Predictions
train_pred = fixed_model.predict(X_train)
val_pred = fixed_model.predict(X_test)

# Scores
fixed_train_score = accuracy_score(
    y_train,
    train_pred
)

fixed_val_score = accuracy_score(
    y_test,
    val_pred
)

fixed_gap = fixed_train_score - fixed_val_score

print("Training Accuracy:", fixed_train_score)
print("Validation Accuracy:", fixed_val_score)
print("Train-Validation Gap:", fixed_gap)

Training Accuracy: 1.0
Validation Accuracy: 1.0
Train-Validation Gap: 0.0


# Task Step 4: Model Fit Diagnosis

## Overfitted Model

The first Decision Tree was intentionally created with unlimited depth.

The model should show a high training score and a lower validation score.

This large train-validation gap is evidence of overfitting.

## Underfitted Model

The second Decision Tree used a very small maximum depth.

The model should show low training and validation performance.

This indicates underfitting because the model is too simple to capture the patterns in the data.

## Fixed Model

The final Decision Tree used a limited maximum depth to reduce model complexity.

The purpose of this change was to reduce overfitting and improve generalization.

The model should be evaluated by comparing its training score, validation score, and train-validation gap with the original overfitted model.

## Conclusion

The train-validation gap is useful for diagnosing model fit.

* Low training and validation scores indicate underfitting.
* High training performance with much lower validation performance indicates overfitting.
* High training and validation performance with a small gap indicates a better fit.

Reducing model complexity is one way to fix overfitting.


In [15]:
print("===== Model Comparison =====")

print("\nOverfitted Model")
print("Train:", overfit_train_score)
print("Validation:", overfit_val_score)
print("Gap:", overfit_gap)

print("\nUnderfitted Model")
print("Train:", underfit_train_score)
print("Validation:", underfit_val_score)
print("Gap:", underfit_gap)

print("\nFixed Model")
print("Train:", fixed_train_score)
print("Validation:", fixed_val_score)
print("Gap:", fixed_gap)

===== Model Comparison =====

Overfitted Model
Train: 1.0
Validation: 1.0
Gap: 0.0

Underfitted Model
Train: 1.0
Validation: 1.0
Gap: 0.0

Fixed Model
Train: 1.0
Validation: 1.0
Gap: 0.0
